# Introduction to JACC.jl
<img src="https://github.com/PhilipFackler/JACC-applications/blob/main/tutorial/images/JACC-logo.png?raw=1" alt="JACC logo" style="width:15%;height:auto;">

## Motivation
Different HPC systems have different hardware and thus different vendor-provided APIs.

Different users have different levels/types of expertise.

Therefore we want:
- performance portability
- programming productivity
- performance "practicability"?

<img src="https://github.com/PhilipFackler/JACC-applications/blob/main/tutorial/images/HPC-systems-vendors.png?raw=1" alt="HPC Systems and Vendors" style="width:90%;height:auto;">

### Options for (data) parallelism in Julia
CPU multi-threading
- Threads (built-in)
- Polyester
- OhMyThreads

GPU kernel model (fine-granularity):
- Vendor-specific packages:
  - CUDA, AMDGPU, oneAPI, Metal
- [KernelAbstractions](https://github.com/JuliaGPU/KernelAbstractions.jl)
  - Provides (mostly) portable interface implemented by each vendor package

### The Aim of JACC

Provide a familiar interface and smart defaults, bringing good performance closer to domain scientists who may not be experts in computer science.

- high-level unified Julia front-end on top of multiple backends (Threads, CUDA, AMDGPU, oneAPI)
- Hide low-level and device-specific implementation
- Improve programming productivity for science and HPC
- A growing community
  - LBNL/NERSC, LANL, Argonne, MIT, ETHZ, BSC, Cerfacs, FI/CCQ, …
  - You are welcome to join (JACC community meetings once a month)
  - Reach out on [github](https://github.com/JuliaORNL/JACC.jl/discussions)

<img src="https://github.com/PhilipFackler/JACC-applications/blob/main/tutorial/images/JACC-stack.png?raw=1" alt="JACC stack" style="width:25%;height:auto;">

## Simple Exercises
1. Allocate an array and use a parallel_for to initialize the elements (e.g., using the index values)
2. Allocate an array of ones and find the sum of the elements using parallel_reduce.
3. Try #2 using a parallel_for with @atomic

(Bonus) Create a struct holding an array and use it in a kernel. (This is not a challenge for CPU-only code)

## Getting started with JACC

##### 1. Add JACC as a dependency

In [ ]:
import Pkg
Pkg.add("JACC")

##### 2. Set backend

The default backend is Threads

In [ ]:
import JACC
@show JACC.backend
@show JACC.supported_backends
@show names(JACC.Backend, all = true)
;

Calling `set_backend` will install the appropriate backend package as a dependency for the current project (don't commit this!).

NOTE: in some cases you may need to configure the backend package before proceeding to use it.

In [ ]:
import JACC
JACC.set_backend("oneapi")
JACC.set_backend(:oneapi)
JACC.set_backend(JACC.Backend.oneapi)

##### 3. Restart Julia
For this notebook, use the menu: "Kernel" > "Restart Kernel..." (With the REPL in a terminal you would simply quit and restart `julia`.)

##### 4. Load extension before use

```julia
JACC.@init_backend
```

Equivalent to
```julia
import <backend-package>
```

In [ ]:
import JACC
JACC.@init_backend

## JACC paradigm basics

### Arrays
There is no separate array type in JACC. Array functions return the backend array type directly.

`JACC.array_type()` will give the corresponding `UnionAll` (non-instantuiated) parametric type.
- useful for function parameters and struct members.

In [ ]:
@show JACC.array_type()
@show JACC.array_type(){Float64, 2}
;

#### Create arrays on-device

- `JACC.ones`
- `JACC.zeros`
- `JACC.fill`
- `JACC.array`

Without a type parameter, these use the default float type (usually `Float64`). You can check with `JACC.default_float()`.

In [ ]:
@show JACC.default_float()

ad_array = JACC.array(3, 2) # uninitialized
println("\nad_array")
display(ad_array)

ad_ones = JACC.ones(4)
println("\nad_ones")
display(ad_ones)

ad_zeros = JACC.zeros(3, 3)
println("\nad_zeros")
display(ad_zeros)

ad_fill = JACC.fill(2.5, 4, 2)
println("\nad_fill")
display(ad_fill)
;

### Transfer

Make a copy

- `JACC.to_device` or `JACC.array(::AbstractArray)`
- `JACC.to_host`

**NOTE**: For `Threads` backend, the above return their argument.

Copy data only (already allocated)

- `JACC.transfer!(dst, src)`

In [ ]:
ah_fill = JACC.to_host(ad_fill)
println("ah_fill")
display(ah_fill)

ah_rand = randn(3,2)
println("\nah_rand")
display(ah_rand)

ad_rand = JACC.to_device(ah_rand)
println("\nad_rand")
display(ad_rand)

JACC.transfer!(ah_rand, ad_array)
println("\nah_rand")
display(ah_rand)
;

### Parallel Kernel Constructs

Primary parameters:
- kernel domain- kernel function
- parameters for kernel function

**NOTE**: kernel function must take thread index(es) as fist parameter(s).

For a kernel function with the signature

```julia
function f(i, x)
    #= 
    ... 
    =#
end
```

#### Function style
- `JACC.parallel_for(N, f, x)`
- `JACC.parallel_reduce(N, f, x)`

#### Macro style
- `JACC.@parallel_for range=N f(x)`
- `JACC.@parallel_reduce range=N f(x)`

**NOTE**: keyword arguments in macro calls cannot use spaces

Example: define `axpy` and `dot` as kernel functions for 1D and 2D arrays:

In [ ]:
# 1D - with classic call style

## define kernel functions
function axpy(i, α, x, y) # element-wise kernel for y = αx + y
    y[i] += α * x[i]
end

function dot(i, x, y)     # element-wise kernel for x ⋅ y
    return x[i] * y[i]
end

## construct arrays
SIZE = 1_000
x = round.(rand(JACC.default_float(), SIZE) * 100)
y = round.(rand(JACC.default_float(), SIZE) * 100)
α = 2.5

xd = JACC.to_device(x)
yd = JACC.to_device(y)

## perform parallel operations
JACC.parallel_for(SIZE, axpy, α, xd, yd)

res = JACC.parallel_reduce(SIZE, dot, xd, yd)

@show res
;

In [ ]:
# 2D - with macro style

## define kernel functions
function axpy(i, j, α, x, y)
    y[i, j] += α * x[i, j]
end

function dot(i, j, x, y)
    return x[i, j] * y[i, j]
end

## construct arrays
X = round.(rand(Float64, SIZE, SIZE) * 100)
Y = round.(rand(Float64, SIZE, SIZE) * 100)

Xd = JACC.to_device(X)
Yd = JACC.to_device(Y)

## perform parallel operations
JACC.@parallel_for range=(SIZE, SIZE) axpy(α, Xd, Yd) # cannot use spaces for keyword arguments

ret = JACC.@parallel_reduce range=(SIZE, SIZE) dot(Xd, Yd)

# macro @parallel_reduce returns an on-device array with one element
res = JACC.to_host(ret)[]

display(ret)
@show res
;

#### Julia `do`-block syntactic sugar

When a function's first parameter is a function, that argument can be skipped and defined in-place using `do ... end`.

Manual [section](https://docs.julialang.org/en/v1/manual/functions/#Do-Block-Syntax-for-Function-Arguments)
uses `map` as example:

```julia
map(f, A)
```

where `f` is a unary function of `f(x)`

```julia
map(A) do x
    # function body defined in-place
end
```

Remember, in JACC, a kernel function must take an index in addition to the other expected parameters.

In [ ]:
### Serial loop
for i = 1:SIZE
    @inbounds y[i] += α * x[i]
end
###

### JACC do-style
JACC.parallel_for(SIZE, α, xd, yd) do i, a, x, y
    @inbounds x[i] += a * y[i]
end
###
;

#### Performance results for simple kernels
<details>
    <summary>Image</summary>
<img src="https://github.com/PhilipFackler/JACC-applications/blob/main/tutorial/images/JACC-perfresults-01.png?raw=1" alt="Performance Results" style="width:90%;height:auto;">
</details>

## [Aside]: kernel arguments

Items must be passed explicitly. 

An implicit reference to a name from the parent scope will be attempting to access host memory from the device (unless you're using the "threads" backend).

Let's see a slightly more involved example and use a named tuple for arguments.

```julia
### Serial loop
for i = 1:length(events)
    @inbounds begin
        event = events[i, 6:8]
        weight = events[i, 1]
        for op in transforms
            v = op * event
            atomic_push!(hist, v, weight)
        end
    end
end

### JACC do-style
JACC.parallel_for(   # named tuple
    length(events), (h = hist, events, transforms)) do i, t
    @inbounds begin
        event = t.events[i, 6:8]
        weight = t.events[i, 1]
        for op in t.transforms
            v = op * event
            atomic_push!(t.h, v, weight)
        end
    end
end
###

### JACC with anonymous function
JACC.parallel_for(length(events),
    (i, t) -> begin
        @inbounds begin
            event = t.events[i, 6:8]
            weight = t.events[i, 1]
            for op in t.transforms
                v = op * event
                atomic_push!(t.h, v, weight)
            end
        end
    end,
    (h = hist, events, transforms), # named tuple
)
###
```

## More API details

### `JACC.parallel_for`

```julia
# idiomatic function application
parallel_for(f, range, x...)

# classic version
parallel_for(range, f, x...)

# keyword arguments version
parallel_for(; range, f, args) # args must be a Tuple

# macro version
@parallel_for range=range f(x...)

```

The `range` can be one of
- `Integer`
- `AbstractRange`
- `NTuple{N, Integer}`
- `NTuple{N, AbstractRange}`

In [ ]:
let
    a = JACC.zeros(10, 10)
    JACC.parallel_for((2:9, 2:9), a) do i, j, a
        a[i, j] += i * j
    end
    display(a)
end

### `JACC.parallel_reduce`

```julia
### idiomatic function application
#
# return: typeof(init)
# defaults:
#  - type: JACC.default_float()
#  - op: +
#  - init: default_init(type, op) # 0.0 for +
parallel_reduce(f, range, x...; [type], [op], [init])

### classic version
parallel_reduce(range, f, x...; [type], [op], [init])

### array-based reduce
#
# return: typeof(init)
# defaults:
#  - op: +
#  - init: default_init(eltype(a), op)
parallel_reduce([op = +,] a::AbstractArray; init = default_init(eltype(a), op))

# op ∊ {+, *, min, max, <user-defined>}

### keyword-arguments version
#
# return: "on-device" single-element array
parallel_reduce(; range, f, args, [type], [op], [init]) # args must be a Tuple

### macro version
#
# return: "on-device" single-element array
@parallel_reduce range=range [type=...] [op=...] [init=...] f(x...)

```

Here are a few simple examples:

In [ ]:
a = JACC.array([i for i in 1:10])
@show JACC.parallel_reduce(a)
@show JACC.parallel_reduce(min, a)

b = JACC.fill(2, 10)
b_min = JACC.parallel_reduce(10, (i, b) -> b[i] + i, b; op = min, init = Inf)
@show b_min
b_max = JACC.@parallel_reduce range=10 op=max init=-Inf JACC.elem_access(b)
@show JACC.to_host(b_max)[]

dp = JACC.parallel_reduce(10, a, b) do i, a, b
    a[i] * b[i]
end
@show dp
;

## Notes for GPU programming
- Any function can be used in a kernel (without special annotation) as long as it doesn't allocate
- JACC imports the `@atomic` macro from [Atomix.jl](https://github.com/JuliaConcurrent/Atomix.jl)
- Use [StaticArrays.jl](https://github.com/JuliaArrays/StaticArrays.jl) for small fixed-size arrays
- Use [Adapt.jl](https://github.com/JuliaGPU/Adapt.jl) if you need to put GPU arrays in a struct